In [ ]:
# ═══════════════════════════════════════════════════════
# MEDICAL RAG - PACKAGE INSTALLATION
# ═══════════════════════════════════════════════════════

# 1. Core RAG framework (pinned version for stability)
!pip install -q langchain==0.0.354

# 2. PDF processing
!pip install -q pypdf

# 3. Embedding model (all-MiniLM-L6-v2)
!pip install -q sentence-transformers

# 4. Vector database
!pip install -q faiss-cpu

# 5. Google Gemini LLM
!pip install -q google-generativeai

print("✅ Installation complete!")
print("📦 Installed packages:")
print("   - langchain (0.0.354)")
print("   - pypdf")
print("   - sentence-transformers")
print("   - faiss-cpu")
print("   - google-generativeai")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.3/803.3 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 241.2/241.2 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.4/55.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph 1.2.4 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.1.23 which is incompatible.
google-adk 1.29.0 requires tenacity<10.0.0,>=9.0.0, but you have tenacity 8.5.0 whic

In [ ]:
import numpy
!pip install --upgrade numpy

# ═══════════════════════════════════════════════════════
# VERIFY INSTALLATION (Run AFTER restart)
# ═══════════════════════════════════════════════════════

import langchain
import pypdf
import sentence_transformers
import faiss
import google.generativeai as genai

print("✅ All imports successful!")
print(f"LangChain version: {langchain.__version__}")
print(f"Sentence Transformers version: {sentence_transformers.__version__}")

✅ All imports successful!
LangChain version: 0.0.354
Sentence Transformers version: 5.5.1


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
# ═══════════════════════════════════════════════════════
# VERIFY FILE STRUCTURE
# ═══════════════════════════════════════════════════════

import os

# Check if Drive is mounted
drive_path = "/content/drive/MyDrive/medical-rag/data"

if os.path.exists(drive_path):
    print("✅ Drive folder found!")
    print("\n📄 Files in your data folder:")
    files = os.listdir(drive_path)
    for file in files:
        file_path = os.path.join(drive_path, file)
        size = os.path.getsize(file_path) / 1024  # KB
        print(f"   - {file} ({size:.2f} KB)")
else:
    print("❌ Drive folder not found!")
    print("Did you mount Google Drive?")

✅ Drive folder found!

📄 Files in your data folder:
   - .ipynb_checkpoints (4.00 KB)
   - paper1.pdf (901.26 KB)
   - paper2.pdf (504.41 KB)


In [ ]:
# ═══════════════════════════════════════════════════════
# TEST GEMINI API CONNECTION
# ═══════════════════════════════════════════════════════

import google.generativeai as genai
from google.colab import userdata

# Get API key from Colab Secrets
try:
    api_key = userdata.get('GEMINI_API_KEY')
    print("✅ API key retrieved from secrets")
except:
    print("❌ Could not find GEMINI_API_KEY in secrets!")
    print("Did you add it and enable notebook access?")
    raise

# Configure Gemini
genai.configure(api_key=api_key)

# Test with a simple medical question
model = genai.GenerativeModel('gemini-2.5-flash')
response = model.generate_content(
    "In one sentence, what is diabetes?"
)

print("\n✅ GEMINI API TEST SUCCESSFUL!")
print("\n🤖 Gemini's response:")
print(response.text)
print("\n" + "="*50)
print("✅ Your LLM is ready for the RAG pipeline!")

✅ API key retrieved from secrets

✅ GEMINI API TEST SUCCESSFUL!

🤖 Gemini's response:
Diabetes is a chronic condition where the body either doesn't produce enough insulin or can't effectively use the insulin it produces, leading to high blood sugar levels.

✅ Your LLM is ready for the RAG pipeline!


In [ ]:
# ═══════════════════════════════════════════════════════
# STEP 4: LOAD MEDICAL PDF
# ═══════════════════════════════════════════════════════

from langchain.document_loaders import PyPDFLoader
import os

# Define path to your first paper
pdf_path = "/content/drive/MyDrive/medical-rag/data/paper1.pdf"

# Verify file exists
if not os.path.exists(pdf_path):
    print(f"❌ File not found: {pdf_path}")
else:
    print(f"✅ Found: paper1.pdf")
    print(f"📊 Size: {os.path.getsize(pdf_path) / 1024:.2f} KB")

# Load the PDF
print("\n🔄 Loading PDF...")
loader = PyPDFLoader(pdf_path)
pages = loader.load()

# Analyze what we loaded
print(f"\n✅ PDF loaded successfully!")
print(f"📄 Total pages: {len(pages)}")
print(f"📝 Total characters: {sum(len(page.page_content) for page in pages):,}")

# Show first page preview
print("\n" + "="*60)
print("📖 FIRST PAGE PREVIEW (first 500 characters):")
print("="*60)
print(pages[0].page_content[:500])
print("...")
print("="*60)

# Show metadata
print("\n📋 Page metadata:")
print(pages[0].metadata)

✅ Found: paper1.pdf
📊 Size: 901.26 KB

🔄 Loading PDF...

✅ PDF loaded successfully!
📄 Total pages: 6
📝 Total characters: 23,138

📖 FIRST PAGE PREVIEW (first 500 characters):
ArchivesofDiseaseinChildhood,1985,60,823-828
Associationofdiabetesinsipidus,diabetesmellitus,
opticatrophy,anddeafness
TheWolframor DIDMOAD syndrome
SSNAJJAR,M G SAIKALY,G M ZAYTOUN, AND A ABDELNOOR
DepartmentsofPediatrics,Otolaryngology,andMicrobiology,AmericanUniversityofBeirutMedical
Center,Beirut,Lebanon
SUMMARY Sevenpatientswitha raresyndromeofdiabetesinsipidus(DI),diabetesmellitus
(DM),opticatrophy(OA),neurosensorydeafness(D),atonyoftheurinarytract,andother
abnormalities(WolframorDIDMOAD s
...

📋 Page metadata:
{'source': '/content/drive/MyDrive/medical-rag/data/paper1.pdf', 'page': 0}


In [ ]:
# ═══════════════════════════════════════════════════════
# STEP 5: CHUNK THE DOCUMENT
# ═══════════════════════════════════════════════════════

from langchain.text_splitter import RecursiveCharacterTextSplitter

# Create text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,        # Each chunk = ~1000 characters
    chunk_overlap=200,      # 200 character overlap between chunks
    length_function=len,    # How to measure length
    separators=["\n\n", "\n", " ", ""]  # Split on paragraphs first, then sentences
)

# Split all pages into chunks
print("🔄 Splitting document into chunks...")
chunks = text_splitter.split_documents(pages)

print(f"\n✅ Document chunked successfully!")
print(f"📄 Original pages: {len(pages)}")
print(f"📦 Total chunks created: {len(chunks)}")
print(f"📊 Average chunk size: {sum(len(c.page_content) for c in chunks) // len(chunks)} characters")

# Show first chunk
print("\n" + "="*60)
print("📦 FIRST CHUNK PREVIEW:")
print("="*60)
print(chunks[0].page_content)
print("\n" + "="*60)

# Show chunk metadata
print("📋 Chunk metadata:")
print(chunks[0].metadata)

🔄 Splitting document into chunks...

✅ Document chunked successfully!
📄 Original pages: 6
📦 Total chunks created: 30
📊 Average chunk size: 909 characters

📦 FIRST CHUNK PREVIEW:
ArchivesofDiseaseinChildhood,1985,60,823-828
Associationofdiabetesinsipidus,diabetesmellitus,
opticatrophy,anddeafness
TheWolframor DIDMOAD syndrome
SSNAJJAR,M G SAIKALY,G M ZAYTOUN, AND A ABDELNOOR
DepartmentsofPediatrics,Otolaryngology,andMicrobiology,AmericanUniversityofBeirutMedical
Center,Beirut,Lebanon
SUMMARY Sevenpatientswitha raresyndromeofdiabetesinsipidus(DI),diabetesmellitus
(DM),opticatrophy(OA),neurosensorydeafness(D),atonyoftheurinarytract,andother
abnormalities(WolframorDIDMOAD syndrome)arereported.Ofthesevenpatients,three
siblingswere followedup for10-17years.
Allsevenpatientshaddiabetesmellitusandopticatrophy;sixhaddiabetesinsipidus;andin
thefourpatientsinvestigatedtherewas dilatationoftheurinarytract.Theseverityofdiabetes
varied,andallrequiredinsulinforcontrolofthehyperglycaemia.Inonepatientt

In [ ]:
# ═══════════════════════════════════════════════════════
# STEP 6: GENERATE EMBEDDINGS
# ═══════════════════════════════════════════════════════

from langchain.embeddings import HuggingFaceEmbeddings

# Initialize embedding model
print("🔄 Loading embedding model (all-MiniLM-L6-v2)...")
print("⏳ This will download ~80 MB on first run...")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},  # Use CPU (GPU not needed for 30 chunks)
    encode_kwargs={'normalize_embeddings': True}  # Normalize for cosine similarity
)

print("✅ Embedding model loaded!")

# Test: Generate embedding for one chunk
print("\n🧪 Testing embedding generation...")
test_text = chunks[0].page_content[:100]  # First 100 chars of first chunk
test_embedding = embeddings.embed_query(test_text)

print(f"✅ Embedding generated successfully!")
print(f"📊 Input text length: {len(test_text)} characters")
print(f"📊 Output vector dimensions: {len(test_embedding)}")
print(f"📊 First 10 dimensions: {test_embedding[:10]}")

print("\n" + "="*60)
print("✅ Embedding model ready to process all 30 chunks!")
print("="*60)

🔄 Loading embedding model (all-MiniLM-L6-v2)...
⏳ This will download ~80 MB on first run...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded!

🧪 Testing embedding generation...
✅ Embedding generated successfully!
📊 Input text length: 100 characters
📊 Output vector dimensions: 384
📊 First 10 dimensions: [-0.001146384747698903, 0.009223678149282932, -0.01075921393930912, 0.0487494058907032, -0.03576111048460007, 0.013377463445067406, 0.0936184972524643, 0.031607892364263535, -0.010129335336387157, 0.033179882913827896]

✅ Embedding model ready to process all 30 chunks!


In [ ]:
# ═══════════════════════════════════════════════════════
# STEP 7: CREATE FAISS VECTOR DATABASE
# ═══════════════════════════════════════════════════════

from langchain.vectorstores import FAISS

print("🔄 Creating FAISS vector database...")
print("⏳ Generating embeddings for all 30 chunks...")

# Create FAISS index from documents
# This will:
# 1. Generate embedding for each chunk
# 2. Store vectors in FAISS index
# 3. Link vectors to original text
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("\n✅ FAISS vector database created!")
print(f"📦 Total chunks indexed: {len(chunks)}")
print(f"📊 Vector dimensions: 384")

# Test: Search for similar content
print("\n" + "="*60)
print("🧪 TEST: Similarity Search")
print("="*60)

test_query = "What are the symptoms of diabetes?"
print(f"🔍 Query: '{test_query}'")
print("\n⏳ Searching for similar chunks...")

# Perform similarity search (returns top 3 most similar chunks)
results = vectorstore.similarity_search(test_query, k=3)

print(f"\n✅ Found {len(results)} relevant chunks:")
for i, doc in enumerate(results, 1):
    print(f"\n📄 Result {i}:")
    print(f"   Source: Page {doc.metadata['page']}")
    print(f"   Content preview: {doc.page_content[:150]}...")

print("\n" + "="*60)
print("✅ VECTOR DATABASE IS WORKING!")
print("✅ You now have a searchable medical knowledge base!")
print("="*60)

🔄 Creating FAISS vector database...
⏳ Generating embeddings for all 30 chunks...

✅ FAISS vector database created!
📦 Total chunks indexed: 30
📊 Vector dimensions: 384

🧪 TEST: Similarity Search
🔍 Query: 'What are the symptoms of diabetes?'

⏳ Searching for similar chunks...

✅ Found 3 relevant chunks:

📄 Result 1:
   Source: Page 3
   Content preview: tion.8
Thediagnosisofdiabetesmellitusprecededthat
ofdiabetesinsipidusinallourpatients.Thisisnota
universalfinding,however;diabetesinsipidushas
beendes...

📄 Result 2:
   Source: Page 0
   Content preview: diabetesmellitus,opticatrophy,andsensorineural
deafness.Hydronephrosis,hydroureters,anddilata-
tionoftheurinarybladderarelesscommonly
associatedfindin...

📄 Result 3:
   Source: Page 4
   Content preview: 5AmosDB,PoolP.HLA-typing.In:RoseNR,FriedmanH,
eds.Manualofclinicalimmunology.2nded.Washington:
AmericanSocietyforMicrobiology,1976:797-804.
6Schuknech...

✅ VECTOR DATABASE IS WORKING!
✅ You now have a searchable medical knowledge base!

In [ ]:
# ═══════════════════════════════════════════════════════
# STEP 8: BUILD RAG QUERY PIPELINE
# ═══════════════════════════════════════════════════════

import google.generativeai as genai
from google.colab import userdata

# Configure Gemini
api_key = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=api_key)
model = genai.GenerativeModel('gemini-2.5-flash')

def ask_medical_rag(question, k=3):
    """
    RAG pipeline: Retrieve relevant chunks and generate answer

    Args:
        question (str): User's medical question
        k (int): Number of chunks to retrieve (default: 3)

    Returns:
        dict: Contains answer, sources, and retrieved chunks
    """

    print(f"🔍 Query: '{question}'")
    print(f"📊 Retrieving top {k} relevant chunks...\n")

    # STEP 1: Retrieve relevant chunks
    retrieved_docs = vectorstore.similarity_search(question, k=k)

    # STEP 2: Build context from retrieved chunks
    context = "\n\n---\n\n".join([
        f"[Source: Page {doc.metadata['page']}]\n{doc.page_content}"
        for doc in retrieved_docs
    ])

    # STEP 3: Build RAG prompt
    prompt = f"""You are a medical assistant. Answer the question based ONLY on the provided context from medical literature.

CONTEXT FROM MEDICAL PAPERS:
{context}

QUESTION: {question}

INSTRUCTIONS:
- Answer based ONLY on the context above
- If the context doesn't contain enough information, say so
- Cite which page the information comes from
- Be concise and accurate
- Use medical terminology appropriately

ANSWER:"""

    # STEP 4: Generate answer with Gemini
    print("🤖 Generating answer with Gemini 2.5 Flash...\n")
    response = model.generate_content(prompt)

    # STEP 5: Return results
    return {
        'question': question,
        'answer': response.text,
        'sources': [f"Page {doc.metadata['page']}" for doc in retrieved_docs],
        'retrieved_chunks': retrieved_docs
    }

# ═══════════════════════════════════════════════════════
# TEST THE COMPLETE RAG SYSTEM
# ═══════════════════════════════════════════════════════

print("="*60)
print("🏥 MEDICAL RAG ASSISTANT - READY!")
print("="*60)
print()

# Test question
test_question = "What are the main symptoms of DIDMOAD syndrome?"

result = ask_medical_rag(test_question)

print("="*60)
print("📋 RAG RESPONSE")
print("="*60)
print(f"\n❓ Question: {result['question']}")
print(f"\n📚 Sources used: {', '.join(result['sources'])}")
print(f"\n💬 Answer:\n{result['answer']}")
print("\n" + "="*60)

🏥 MEDICAL RAG ASSISTANT - READY!

🔍 Query: 'What are the main symptoms of DIDMOAD syndrome?'
📊 Retrieving top 3 relevant chunks...

🤖 Generating answer with Gemini 2.5 Flash...

📋 RAG RESPONSE

❓ Question: What are the main symptoms of DIDMOAD syndrome?

📚 Sources used: Page 4, Page 0, Page 1

💬 Answer:
The main symptoms of DIDMOAD syndrome include diabetes insipidus, diabetes mellitus, optic atrophy, neurosensory deafness, and atony or dilatation of the urinary tract (Page 0, Page 1).



In [ ]:
# ═══════════════════════════════════════════════════════
# STEP 9: TEST MULTIPLE QUESTIONS
# ═══════════════════════════════════════════════════════

print("="*60)
print("🧪 TESTING RAG SYSTEM WITH MULTIPLE QUESTIONS")
print("="*60)
print()

# Test questions covering different aspects
test_questions = [
    "What is the treatment for diabetes in DIDMOAD patients?",
    "How many patients were studied in this research?",
    "What causes quantum entanglement?",  # Out of scope - should say "not in context"
]

for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*60}")
    print(f"TEST {i}/{len(test_questions)}")
    print(f"{'='*60}")

    result = ask_medical_rag(question, k=3)

    print(f"\n💬 Answer:\n{result['answer']}")
    print(f"\n📚 Sources: {', '.join(result['sources'])}")
    print()

print("="*60)
print("✅ TESTING COMPLETE")
print("="*60)

🧪 TESTING RAG SYSTEM WITH MULTIPLE QUESTIONS


TEST 1/3
🔍 Query: 'What is the treatment for diabetes in DIDMOAD patients?'
📊 Retrieving top 3 relevant chunks...

🤖 Generating answer with Gemini 2.5 Flash...


💬 Answer:
All DIDMOAD patients required insulin for control of hyperglycemia (Page 0, Page 1).

📚 Sources: Page 4, Page 1, Page 0


TEST 2/3
🔍 Query: 'How many patients were studied in this research?'
📊 Retrieving top 3 relevant chunks...

🤖 Generating answer with Gemini 2.5 Flash...


💬 Answer:
Seven patients were studied in this research (Page 1, Table 1).

📚 Sources: Page 1, Page 2, Page 2


TEST 3/3
🔍 Query: 'What causes quantum entanglement?'
📊 Retrieving top 3 relevant chunks...

🤖 Generating answer with Gemini 2.5 Flash...


💬 Answer:
The provided context does not contain information about the causes of quantum entanglement.

📚 Sources: Page 2, Page 4, Page 5

✅ TESTING COMPLETE


In [ ]:
# ═══════════════════════════════════════════════════════
# STEP 10: EXPAND KNOWLEDGE BASE WITH PAPER 2
# ═══════════════════════════════════════════════════════

print("🔄 Loading second medical paper...")

# Load paper2.pdf
pdf_path_2 = "/content/drive/MyDrive/medical-rag/data/paper2.pdf"
loader_2 = PyPDFLoader(pdf_path_2)
pages_2 = loader_2.load()

print(f"✅ Paper 2 loaded: {len(pages_2)} pages")

# Chunk paper 2
chunks_2 = text_splitter.split_documents(pages_2)
print(f"📦 Paper 2 chunks: {len(chunks_2)}")

# Combine with paper 1 chunks
all_chunks = chunks + chunks_2
print(f"\n📊 Total chunks: {len(all_chunks)}")
print(f"   - Paper 1: {len(chunks)} chunks")
print(f"   - Paper 2: {len(chunks_2)} chunks")

# Rebuild FAISS with both papers
print("\n🔄 Rebuilding FAISS index with both papers...")
vectorstore_expanded = FAISS.from_documents(
    documents=all_chunks,
    embedding=embeddings
)

print("✅ Expanded knowledge base ready!")
print(f"📚 Now searchable: {len(all_chunks)} chunks from 2 papers")

# Update the global vectorstore
vectorstore = vectorstore_expanded

🔄 Loading second medical paper...
✅ Paper 2 loaded: 4 pages
📦 Paper 2 chunks: 18

📊 Total chunks: 48
   - Paper 1: 30 chunks
   - Paper 2: 18 chunks

🔄 Rebuilding FAISS index with both papers...
✅ Expanded knowledge base ready!
📚 Now searchable: 48 chunks from 2 papers


In [ ]:
# ═══════════════════════════════════════════════════════
# TEST EXPANDED KNOWLEDGE BASE
# ═══════════════════════════════════════════════════════

print("="*60)
print("🧪 TESTING EXPANDED RAG SYSTEM (2 PAPERS)")
print("="*60)
print()

# Test questions that might pull from different papers
test_questions = [
    "What are the main symptoms of DIDMOAD syndrome?",  # Should cite Paper 1
    "What treatment options are discussed in the papers?",  # Might cite both papers
    "How is quantum physics related to medicine?",  # Out of scope test
]

for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*60}")
    print(f"TEST {i}/{len(test_questions)}")
    print(f"{'='*60}")

    result = ask_medical_rag(question, k=3)

    print(f"\n💬 Answer:\n{result['answer']}")
    print(f"\n📚 Sources: {', '.join(result['sources'])}")

    # Show which chunks were retrieved
    print(f"\n📄 Retrieved from:")
    for doc in result['retrieved_chunks']:
        print(f"   - Page {doc.metadata['page']}: {doc.page_content[:80]}...")
    print()

print("="*60)
print("✅ TESTING COMPLETE")
print("="*60)

🧪 TESTING EXPANDED RAG SYSTEM (2 PAPERS)


TEST 1/3
🔍 Query: 'What are the main symptoms of DIDMOAD syndrome?'
📊 Retrieving top 3 relevant chunks...

🤖 Generating answer with Gemini 2.5 Flash...


💬 Answer:
The main symptoms of DIDMOAD syndrome are diabetes insipidus (DI), diabetes mellitus (DM), optic atrophy (OA), neurosensory deafness (D), and atony of the urinary tract (Page 0).

📚 Sources: Page 4, Page 0, Page 0

📄 Retrieved from:
   - Page 4: statusinfamilywith(Diabetesinsipidusandmellitus,optic
atrophyanddeafness)DIDMOAD...
   - Page 0: ArchivesofDiseaseinChildhood,1985,60,823-828
Associationofdiabetesinsipidus,diab...
   - Page 0: thatitrepresentsasyndrome(Roseetal.,1966;
Jeanetal.,1970).Thesyndromeoccursinsib...


TEST 2/3
🔍 Query: 'What treatment options are discussed in the papers?'
📊 Retrieving top 3 relevant chunks...

🤖 Generating answer with Gemini 2.5 Flash...


💬 Answer:
The provided context does not discuss any treatment options (Page 1, Page 2).

📚 Sources: Page 2, P

In [ ]:
# ═══════════════════════════════════════════════════════
# FINAL: INTERACTIVE MEDICAL RAG ASSISTANT
# ═══════════════════════════════════════════════════════

def medical_assistant_demo():
    """
    Interactive demo of the Medical RAG Assistant
    """

    print("╔════════════════════════════════════════════════════════╗")
    print("║     🏥 MEDICAL RAG ASSISTANT - INTERACTIVE DEMO 🏥     ║")
    print("╚════════════════════════════════════════════════════════╝")
    print()
    print("📚 Knowledge Base: 2 Medical Papers (48 chunks)")
    print("🤖 Powered by: Gemini 2.5 Flash + FAISS")
    print()
    print("="*60)
    print()

    # Example questions
    example_questions = [
        "What are the main symptoms of DIDMOAD syndrome?",
        "How many patients were studied?",
        "What is the relationship between diabetes insipidus and diabetes mellitus in DIDMOAD?",
        "What treatment is recommended for optic atrophy?",
    ]

    print("💡 Example Questions:")
    for i, q in enumerate(example_questions, 1):
        print(f"   {i}. {q}")
    print()
    print("="*60)

    # Interactive loop
    while True:
        print()
        question = input("❓ Your Question (or 'quit' to exit): ").strip()

        if question.lower() in ['quit', 'exit', 'q']:
            print("\n👋 Thank you for using Medical RAG Assistant!")
            break

        if not question:
            print("⚠️  Please enter a question.")
            continue

        print()
        print("🔄 Processing...")

        try:
            result = ask_medical_rag(question, k=3)

            print("\n" + "="*60)
            print("💬 ANSWER:")
            print("="*60)
            print(result['answer'])
            print()
            print(f"📚 Sources: {', '.join(set(result['sources']))}")
            print("="*60)

        except Exception as e:
            print(f"❌ Error: {e}")
            print("Please try a different question.")

# Run the demo
medical_assistant_demo()

╔════════════════════════════════════════════════════════╗
║     🏥 MEDICAL RAG ASSISTANT - INTERACTIVE DEMO 🏥     ║
╚════════════════════════════════════════════════════════╝

📚 Knowledge Base: 2 Medical Papers (48 chunks)
🤖 Powered by: Gemini 2.5 Flash + FAISS


💡 Example Questions:
   1. What are the main symptoms of DIDMOAD syndrome?
   2. How many patients were studied?
   3. What is the relationship between diabetes insipidus and diabetes mellitus in DIDMOAD?
   4. What treatment is recommended for optic atrophy?


❓ Your Question (or 'quit' to exit): How many patients were studied?

🔄 Processing...
🔍 Query: 'How many patients were studied?'
📊 Retrieving top 3 relevant chunks...

🤖 Generating answer with Gemini 2.5 Flash...


💬 ANSWER:
Seven patients were studied (cases 1, 2, 3, 4, 5, 6, and 7) (Page 2).

📚 Sources: Page 1, Page 2

❓ Your Question (or 'quit' to exit): what is DIDMOAD syndrome?

🔄 Processing...
🔍 Query: 'what is DIDMOAD syndrome?'
📊 Retrieving top 3 relevant chunks